In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import uuid
import json
import joblib
from cuml.ensemble import RandomForestClassifier  # GPU-accelerated sklearn-like RF
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [3]:
def _load_kmeans_tree_node(node_path):
    # ---------- Load metadata ----------
    with open(os.path.join(node_path, "node.json"), "r") as f:
        metadata = json.load(f)

    node_id = metadata["node_id"]
    children_ids = metadata["children_ids"]

    # ---------- Load model ----------
    model_path = os.path.join(node_path, "model.joblib")
    model = joblib.load(model_path) if os.path.exists(model_path) else None

    # ---------- Load random forest ----------
    rf_path = os.path.join(node_path, "forest_model/rf_model.joblib")
    if os.path.exists(rf_path):
        rf = joblib.load(rf_path)
    else:
        rf = None

    # ---------- Load children ----------
    children = []
    for cid in children_ids:
        child_path = os.path.join(node_path, cid)
        children.append(_load_kmeans_tree_node(child_path))

    # ---------- Rebuild dictionary ----------
    return {
        "level": metadata["level"],
        "node_id": node_id,
        "n_samples": metadata["n_samples"],
        "k": metadata["k"],
        "kmeans_model": model,
        "children": children,
        "stop_reason": metadata["stop_reason"],
        "rf": rf,
    }

def load_kmeans_tree(base_path):
    """
    Loads the full KMeans tree saved by save_kmeans_tree().
    Returns the tree dictionary.
    """

    # Find root (only directory at base_path)
    candidates = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    if len(candidates) != 1:
        raise ValueError("Ambiguous root folder. base_path must contain exactly one folder for the root node.")

    root_id = candidates[0]
    root_path = os.path.join(base_path, root_id)

    return _load_kmeans_tree_node(root_path)

def load_leaf_data(base_path, node_id):
    """
    Loads (X, y) for a specific leaf node using its node_id.
    """

    # Walk through directories searching for node_id
    for root, dirs, files in os.walk(base_path):
        if os.path.basename(root) == node_id:
            data_path = os.path.join(root, "data.joblib")
            if not os.path.exists(data_path):
                raise ValueError(f"Node {node_id} exists but has no leaf data.")

            return joblib.load(data_path)

    raise ValueError(f"Node {node_id} not found under {base_path}.")

def load_leaf_path(base_path, node_id):
    """
    Gives full path for a specific leaf node using its node_id.
    """

    # Walk through directories searching for node_id
    for root, dirs, files in os.walk(base_path):
        if os.path.basename(root) == node_id:
            data_path = os.path.join(root, "data.joblib")
            if not os.path.exists(data_path):
                raise ValueError(f"Node {node_id} exists but has no leaf data.")

            return root

    raise ValueError(f"Node {node_id} not found under {base_path}.")

In [6]:
def save_leaf_zero_var_cols(base_path, zero_var_filename="zero_var_cols.joblib"):
    """
    Finds all leaf nodes in the tree structure under base_path,
    loads their data (X), identifies columns with 1 or fewer unique values,
    and saves the list of these column names to a file inside the
    'forest_model' subfolder of each leaf node's directory.
    """

    print(f"Starting scan for leaf nodes under: {base_path}")
    leaf_ids = []

    # 1. First pass: Identify all leaf node IDs by checking for 'data.joblib'
    for root, dirs, files in os.walk(base_path):
        if "data.joblib" in files:
            leaf_ids.append(os.path.basename(root))

    if not leaf_ids:
        print("No leaf nodes (directories containing 'data.joblib') found.")
        return

    print(f"Found {len(leaf_ids)} leaf nodes to process.")

    # 2. Second pass: Process each leaf node
    for i, leaf_id in enumerate(leaf_ids):
        print(f"Processing leaf {i+1}/{len(leaf_ids)}: {leaf_id}...")
        try:
            # Load the path to the leaf node folder
            leaf_path = load_leaf_path(base_path, leaf_id)

            # Load the data (X, y tuple)
            X, y = load_leaf_data(base_path, leaf_id)

            # Convert X to a pandas DataFrame
            # Assumes X is a NumPy array or similar structure compatible with DataFrame creation
            X_df = pd.DataFrame(X)

            # Find columns with 1 or fewer unique values
            # nunique() returns a Series of counts, <= 1 selects the columns (Series of bools)
            # .index gives the column names where the condition is True
            zero_var_cols = list(X_df.columns[X_df.nunique(axis=0, dropna=True) <= 1])

            # Define the output directory and file path
            output_dir = os.path.join(leaf_path, "forest_model")
            output_file = os.path.join(output_dir, zero_var_filename)

            # Create the output directory if it doesn't exist
            os.makedirs(output_dir, exist_ok=True)

            # Save the list of zero variance columns using joblib
            joblib.dump(zero_var_cols, output_file)
            print(f"  -> Saved {len(zero_var_cols)} zero-variance columns to: {output_file}")

        except ValueError as e:
            print(f"  -> Error processing leaf {leaf_id}: {e}")
        except Exception as e:
            print(f"  -> An unexpected error occurred while processing leaf {leaf_id}: {e}")

    print("Finished processing all leaf nodes.")

# Example usage (assuming necessary imports and helper functions are defined):
# save_leaf_zero_var_cols("/path/to/your/model/tree")

In [7]:
# -----------------------------
# 1. Load data from a leaf
# -----------------------------
drive.mount('/content/drive')

model_path = "/content/drive/Shared drives/Gestió de Projectes/Projecte/saved_model"
save_leaf_zero_var_cols(model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting scan for leaf nodes under: /content/drive/Shared drives/Gestió de Projectes/Projecte/saved_model
Found 36 leaf nodes to process.
Processing leaf 1/36: 39e70ff1-e4ca-4e26-aa38-3ed8f275e0c7...
  -> Saved 2 zero-variance columns to: /content/drive/Shared drives/Gestió de Projectes/Projecte/saved_model/10f65179-dd9c-4d6c-9312-e266dceec286/4cb6f42d-b934-4761-a758-fe2fc39aaf11/b7a0d59d-c9ee-4a98-99bc-fe748abc8160/aa6440b9-9ddb-407b-a976-a08b75e80ac5/9d024d59-a0e2-4e0c-a46a-8b2862df73ec/c6f08e8a-f0e5-4f92-b06c-7d7a877dc1f0/6a3fe048-dee2-4995-a80a-bee79babd559/39e70ff1-e4ca-4e26-aa38-3ed8f275e0c7/forest_model/zero_var_cols.joblib
Processing leaf 2/36: 5162d10f-0fae-4f5c-a573-d92b56191f42...
  -> Saved 2 zero-variance columns to: /content/drive/Shared drives/Gestió de Projectes/Projecte/saved_model/10f65179-dd9c-4d6c-9312-e266dceec286/4cb6f42d-b934-4761-a758-